# DSCO 2025: The Great Data Heist - Phishing URL Detection

**Team Rokunana**
- Rahardi Salim
- Christian Yudistira Hermawan
- Sean Farrel

## Competition Overview
This notebook contains a complete solution for the Data Science Olympiads Competition 2025.

**Theme:** "The Great Data Heist: Unmasking the Hidden Truth"

**Objective:** Develop robust Machine Learning models to distinguish between legitimate and phishing URLs using the LegitPhish dataset.

**Evaluation Metric:** ROC-AUC

---

## Notebook Structure
1. **Part 1: Feature Extraction** - Extract additional features from URLs using web scraping
2. **Part 2: Feature Engineering & Model Training** - Build and train CatBoost classifier
3. **Part 3: Main Submission Generation** - Single model approach (Primary)
4. **Part 4: Alternative Ensemble Approach** - Multi-model ensemble with weighted averaging (Optional)

---
# Part 1: Feature Extraction from URLs

This section extracts additional features from URLs such as:
- HTTP status codes
- Accessibility checks
- Login form detection
- iFrame presence
- Redirect detection
- SSL errors
- External link ratios
- Content length

In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.parse import urlparse
import urllib3
import concurrent.futures
from tqdm.auto import tqdm # Gunakan tqdm.auto biar bagus di Notebook Kaggle
import os

# 1. Supaya log bersih dari warning SSL
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [ ]:
def get_url_features_worker(url):
    """
    Fungsi worker untuk ekstraksi fitur dari 1 URL.
    Mengembalikan Dictionary fitur.
    """
    # Fitur Default (Nilai jika URL mati/error)
    features = {
        'http_status': 0,
        'is_accessible': 0,
        'has_login_form': 0,
        'has_iframe': 0,
        'is_redirected': 0,
        'ssl_error': 0,
        'external_link_ratio': 0.0,
        'content_length': 0
    }

    # User-Agent wajib biar gak dianggap bot script
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36'
    }

    try:
        # TIMEOUT: (Connect Timeout=3s, Read Timeout=5s)
        # verify=False: Abaikan sertifikat SSL (karena phishing sering invalid)
        response = requests.get(url, headers=headers, timeout=(3, 5), verify=False)
        
        # --- Basic Network Features ---
        features['is_accessible'] = 1
        features['http_status'] = response.status_code
        features['content_length'] = len(response.content)
        
        # Cek Redirect (Jika URL di browser beda dgn URL input)
        if response.history or (response.url != url):
            features['is_redirected'] = 1

        # --- Content/HTML Features ---
        soup = BeautifulSoup(response.text, 'html.parser')

        # 1. Cek Form Login (Input Type Password)
        if soup.find('input', {'type': 'password'}):
            features['has_login_form'] = 1
            
        # 2. Cek Iframe
        if soup.find('iframe'):
            features['has_iframe'] = 1

        # 3. Cek Rasio External Link (Domain Mismatch)
        links = soup.find_all('a', href=True)
        total_links = len(links)
        external_links = 0
        
        # Ambil domain dari URL akhir (setelah redirect)
        current_domain = urlparse(response.url).netloc
        
        for link in links:
            try:
                href_parsed = urlparse(link['href'])
                href_domain = href_parsed.netloc
                
                # Logic: Jika ada domain di href, dan domain itu BEDA dengan domain halaman saat ini
                # (Sering terjadi di phishing: Halaman login palsu, tapi link 'About Us' mengarah ke Bank asli)
                if href_domain and href_domain != current_domain:
                    external_links += 1
            except:
                continue
        
        if total_links > 0:
            features['external_link_ratio'] = external_links / total_links

    except requests.exceptions.SSLError:
        features['ssl_error'] = 1
    except Exception:
        # Error lain (Timeout, Connection Refused, DNS Error) biarkan default features (0)
        pass 

    return features

In [ ]:
def process_kaggle_safe(df, url_col='URL', output_filename='processed_features.csv', batch_size=50, max_workers=50):
    """
    Fungsi utama dengan Batch Processing & Resume Capability.
    """
    urls = df[url_col].tolist()
    total_urls = len(urls)
    output_path = f"./{output_filename}" # Save di current working directory
    
    # --- LOGIC RESUME (Lanjut dari yang putus) ---
    start_index = 0
    if os.path.exists(output_path):
        try:
            # Cek berapa baris yang sudah tersimpan
            df_exist = pd.read_csv(output_path)
            start_index = len(df_exist)
            print(f"🔄 File '{output_filename}' ditemukan. Melanjutkan dari baris ke-{start_index}...")
        except pd.errors.EmptyDataError:
            print("⚠️ File ada tapi kosong/rusak. Mulai dari awal.")
            pass
    else:
        print("🚀 Memulai proses baru...")
    
    if start_index >= total_urls:
        print("✅ Semua data sudah selesai diproses!")
        return pd.read_csv(output_path)

    # --- MAIN LOOP PER BATCH ---
    print(f"Processing {total_urls - start_index} URLs remaining...")
    
    # Loop range(start, end, step)
    for i in range(start_index, total_urls, batch_size):
        end_idx = min(i + batch_size, total_urls)
        
        # Slice URL untuk batch ini
        batch_urls = urls[i:end_idx]
        
        # Processing Parallel
        batch_results = []
        with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
            # Gunakan list() untuk memicu eksekusi
            # tqdm untuk progress bar per batch
            batch_results = list(tqdm(executor.map(get_url_features_worker, batch_urls), 
                                      total=len(batch_urls), 
                                      desc=f"Batch {i}-{end_idx}", 
                                      leave=False))
        
        # Buat DataFrame hasil fitur batch ini
        df_features_batch = pd.DataFrame(batch_results)
        
        # Ambil DataFrame ASLI yang sesuai index batch ini (PENTING: reset_index drop=True)
        # Kita pakai iloc supaya presisi mengambil baris yang sesuai
        df_original_batch = df.iloc[i:end_idx].reset_index(drop=True)
        
        # Gabungkan Data Asli + Fitur Baru (Side by Side)
        df_final_batch = pd.concat([df_original_batch, df_features_batch], axis=1)
        
        # --- SAVE TO CSV (APPEND MODE) ---
        # Header hanya ditulis jika file baru dibuat (i == 0 dan start_index == 0)
        # Atau jika file belum ada
        write_header = (not os.path.exists(output_path))
        
        df_final_batch.to_csv(output_path, mode='a', index=False, header=write_header)
        
        # Optional: Print status
        # print(f"✅ Batch {i} - {end_idx} saved.")

    print(f"\n🎉 Selesai! Hasil tersimpan di: {output_path}")
    return pd.read_csv(output_path)

## Run Feature Extraction

**Note:** This process can take a long time depending on the number of URLs. The function supports resume capability, so you can stop and restart without losing progress.

In [ ]:
# Load original train and test data
print("Loading original datasets...")
df_train = pd.read_csv('/kaggle/input/dsco/train.csv')
df_test = pd.read_csv('/kaggle/input/dsco/test.csv')

print(f"Train shape: {df_train.shape}")
print(f"Test shape: {df_test.shape}")

In [ ]:
# # Extract features for TRAIN set
# print("\n" + "="*70)
# print("EXTRACTING FEATURES FOR TRAIN SET")
# print("="*70)

# df_train_featured = process_kaggle_safe(
#     df_train, 
#     url_col='URL', 
#     output_filename='train_with_features.csv',
#     batch_size=50,
#     max_workers=50
# )

# print(f"\nTrain with features shape: {df_train_featured.shape}")
# df_train_featured.head()

In [ ]:
# # Extract features for TEST set
# print("\n" + "="*70)
# print("EXTRACTING FEATURES FOR TEST SET")
# print("="*70)

# df_test_featured = process_kaggle_safe(
#     df_test, 
#     url_col='URL', 
#     output_filename='test_with_features.csv',
#     batch_size=50,
#     max_workers=50
# )

# print(f"\nTest with features shape: {df_test_featured.shape}")
# df_test_featured.head()

---
# Part 2: Feature Engineering & Model Training

This section performs:
1. Leak-free feature engineering using fit/transform pattern
2. CatBoost model training with Optuna hyperparameter tuning
3. 5-Fold Cross-Validation
4. Feature importance analysis

In [ ]:
import pandas as pd
import numpy as np
import urllib.parse
import math
import re
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
import optuna

from catboost import CatBoostClassifier, Pool
from sklearn.preprocessing import MinMaxScaler, QuantileTransformer
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.INFO)

## Feature Engineering Pipeline

This pipeline ensures NO DATA LEAKAGE by:
- Learning statistics (bin edges) ONLY from training data
- Applying the same transformations to test data

In [ ]:
# ============================================================================
# 1. LEAK-FREE FEATURE ENGINEERING PIPELINE (FIT/TRANSFORM)
# ============================================================================
class PhishingFeatureEngineer:
    def __init__(self):
        # We will store statistics learned from the TRAIN set here
        self.len_bin_edges = None
        self.fitted = False
        
    def fit(self, df):
        """
        Learns statistics (bin edges) ONLY from the Training data.
        """
        print("    🔎 [Fit] Learning statistics from Training data...")
        
        # Learn bin edges for URL length based on Train distribution
        # retbins=True returns (binned_series, bins_array)
        _, self.len_bin_edges = pd.cut(df['url_length'], bins=10, retbins=True)
        
        self.fitted = True
        return self

    def transform(self, df):
        """
        Applies feature engineering using the learned statistics.
        Safe for both Train and Test.
        """
        if not self.fitted:
            raise ValueError("⚠️ Engineer must be fitted on Train data first!")
            
        df = df.copy()
        
        # --- 1. URL Component Extraction ---
        df['URL'] = df['URL'].astype(str)
        df['protocol'] = df['URL'].apply(lambda x: 'https' if 'https://' in x else 'http')
        df['domain'] = df['URL'].apply(lambda x: x.split('/')[2] if '//' in x else x)
        df['path'] = df['URL'].apply(lambda x: '/'.join(x.split('/')[3:]) if '//' in x else '')

        # --- 2. Structural Features ---
        df['url_length_ratio'] = df['url_length'] / 100
        df['is_long_url'] = (df['url_length'] > 54).astype(int)
        df['domain_length_ratio'] = df['domain_name_length'] / 30
        df['suspicious_domain_length'] = (df['domain_name_length'] > 25).astype(int)
        df['has_path'] = (df['path_length'] > 0).astype(int)
        df['path_length_ratio'] = np.where(df['path_length'] > 0, 
                                           df['path_length'] / df['url_length'], 0)
        df['has_query_params'] = (df['query_param_count'] > 0).astype(int)
        df['suspicious_query_params'] = (df['query_param_count'] > 3).astype(int)
        df['dot_count_ratio'] = df['dot_count'] / df['url_length']
        df['dots_in_domain'] = df['dot_count'] - 1

        # --- 3. Domain Features ---
        df['tld_length_ratio'] = df['tld_length'] / 10
        df['suspicious_tld'] = (df['tld_popularity'] == 0).astype(int)
        df['has_subdomain'] = (df['subdomain_count'] > 0).astype(int)
        df['multiple_subdomains'] = (df['subdomain_count'] > 2).astype(int)
        df['subdomain_count_ratio'] = df['subdomain_count'] / 5
        df['ip_only_url'] = df['has_ip_address']

        # --- 4. Entropy Features ---
        df['high_entropy'] = (df['url_entropy'] > 4.5).astype(int)
        df['entropy_ratio'] = df['url_entropy'] / 5.5
        df['entropy_density'] = df['url_entropy'] / (df['url_length'] + 1)
        df['is_very_high_entropy'] = (df['url_entropy'] > 4.5).astype(int)
        df['entropy_digit_ratio'] = df['url_entropy'] / (df['number_of_digits'] + 1)
        df['high_numeric_percentage'] = (df['percentage_numeric_chars'] > 30).astype(int)
        df['avg_token_length'] = np.where(df['token_count'] > 0,
                                          df['url_length'] / df['token_count'], 0)

        # --- 5. Robust (Binned) Features - NO LEAKAGE ---
        # We use the bins learned from Train. 
        # We adjust the first and last bin to -inf and +inf to handle outliers in Test.
        
        robust_len_bins = self.len_bin_edges.copy()
        robust_len_bins[0] = -np.inf
        robust_len_bins[-1] = np.inf
        
        df['len_bin'] = pd.cut(df['url_length'], bins=robust_len_bins, labels=False).fillna(-1).astype(str)
        df['entropy_bin'] = df['url_entropy'].round(1).astype(str)
        df['entropy_x_len'] = df['entropy_bin'] + "_" + df['len_bin']

        # --- 6. Security Features ---
        df['no_https'] = (df['https_flag'] == 0).astype(int)
        df['security_score'] = (df['https_flag'] + 
                               (1 - df['suspicious_file_extension']) +
                               (1 - df['has_ip_address'])) / 3
        
        return df

## Data Loading & Processing

In [ ]:
# ==============================================================================
# 2. DATA LOADING & PROCESSING STRATEGY (SEPARATE STREAMS)
# ==============================================================================
def load_and_process_data():
    print("\n📥 Loading data...")
    train_df = pd.read_csv('/kaggle/input/dsbaru-request/train_with_features.csv')
    test_df = pd.read_csv('/kaggle/input/dsbaru-request/test_with_features.csv')
    
    # Initialize Engineer
    engineer = PhishingFeatureEngineer()
    
    # 1. FIT (Learn) on Train
    print("🎓 Fitting Feature Engineer on TRAIN data...")
    engineer.fit(train_df)
    
    # 2. TRANSFORM Train
    print("⚙️ Transforming TRAIN data...")
    train_fe = engineer.transform(train_df)
    
    # 3. TRANSFORM Test (Using Train stats)
    print("⚙️ Transforming TEST data...")
    test_fe = engineer.transform(test_df)
    
    # Handle Categoricals (Common handling)
    cat_cols = ['protocol', 'domain', 'path', 'entropy_bin', 'len_bin', 'entropy_x_len']
    
    for c in cat_cols:
        # Fill missing with 'missing' and ensure string type
        if c in train_fe.columns:
            train_fe[c] = train_fe[c].fillna('missing').astype(str)
            test_fe[c] = test_fe[c].fillna('missing').astype(str)

    # Prepare X and y
    X = train_fe.drop(['URL', 'ClassLabel'], axis=1)
    y = train_fe['ClassLabel'].astype(int)
    
    # Prepare X_test (Ensure alignment)
    X_test = test_fe.drop(['URL', 'ClassLabel'], axis=1, errors='ignore')
    
    print(f"\n✅ Data Ready. Train: {X.shape}, Test: {X_test.shape}")
    return X, y, X_test, cat_cols

## Feature Importance Visualization

In [ ]:
# ==============================================================================
# 3. HELPER: PLOT FEATURE IMPORTANCE
# ==============================================================================
def plot_feature_importance(model, feature_names):
    importances = model.get_feature_importance()
    
    fi_df = pd.DataFrame({
        'feature': feature_names,
        'importance': importances
    })
    
    fi_df = fi_df.sort_values(by='importance', ascending=False).head(20)
    
    plt.figure(figsize=(10, 8))
    sns.barplot(x='importance', y='feature', data=fi_df, palette='viridis')
    plt.title('Top 20 Features - CATBOOST')
    plt.xlabel('Importance Score')
    plt.ylabel('Feature')
    plt.tight_layout()
    plt.show()
    
    print(f"\n🏆 Top 10 Features for CATBOOST:")
    print(fi_df[['feature', 'importance']].head(10).to_string(index=False))
    print("-" * 40)

## Model Training with Optuna Hyperparameter Tuning

In [ ]:
# ==============================================================================
# 4. TUNING & TRAINING
# ==============================================================================

def train_catboost_optimized(X, y, X_test, cat_features=None, n_trials=10):
    # Cross Validation Split
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    test_pred = np.zeros(len(X_test))
    
    last_model = None 

    print(f"\n🔎 Tuning CATBOOST (Optuna with 5-Fold CV)...")

    # --- Step 1: Optuna Optimization ---
    def objective(trial):
        params = {
            'iterations': trial.suggest_int('iterations', 1000, 3000),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'depth': trial.suggest_int('depth', 4, 10),
            'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 0.1, 10.0),
            'random_strength': trial.suggest_float('random_strength', 1e-9, 10.0, log=True),
            'loss_function': 'Logloss', 
            'eval_metric': 'AUC', 
            'verbose': False, 
            'random_seed': 42,
            'allow_writing_files': False,
            'task_type': 'CPU'
        }

        cv_scores = []
        
        # Loop through folds
        for train_idx, val_idx in skf.split(X, y):
            X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

            model = CatBoostClassifier(**params)
            model.fit(X_tr, y_tr, 
                      eval_set=(X_val, y_val), 
                      cat_features=cat_features, 
                      early_stopping_rounds=50,
                      verbose=False)
            
            pred = model.predict_proba(X_val)[:, 1]
            score = roc_auc_score(y_val, pred)
            cv_scores.append(score)

        return np.mean(cv_scores)

    # # UNCOMMENT THIS BLOCK IF YOU WANT TO RE-TUNE
    # study = optuna.create_study(direction='maximize')
    # study.optimize(objective, n_trials=n_trials)
    # print(f"\n[CATBOOST] Best Params found: {study.best_params}")
    # best_params = study.best_params
    
    # DEFAULT HIGH PERFORMING PARAMS (To save time)
    best_params = {'iterations': 1203, 'learning_rate': 0.04450144035178389, 'depth': 8, 'l2_leaf_reg': 4.550408789188565, 'random_strength': 8.369344785680863e-09}
    # # Add static params
    # best_params.update({
    #     'loss_function': 'Logloss', 
    #     'eval_metric': 'AUC', 
    #     'verbose': False, 
    #     'random_seed': 42, 
    #     'allow_writing_files': False
    # })

    # --- Step 2: Final Training with Best Params ---
    print(f"\n🚀 Training Final Models with Best Params...")
    
    fold_scores = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

        model = CatBoostClassifier(**best_params)
        model.fit(
            X_tr, y_tr, 
            eval_set=(X_val, y_val), 
            cat_features=cat_features, 
            early_stopping_rounds=150, 
            verbose=False
        )
        
        # Calculate Fold Score
        val_pred = model.predict_proba(X_val)[:, 1]
        score = roc_auc_score(y_val, val_pred)
        fold_scores.append(score)
        print(f"   Fold {fold+1}: AUC = {score:.6f}")

        # Predict Test Set (Averaging across folds)
        test_pool = Pool(X_test, cat_features=cat_features)
        test_pred += model.predict_proba(test_pool)[:, 1] / skf.n_splits
        
        last_model = model

    print(f"\n✅ Mean Final CV Score: {np.mean(fold_scores):.6f}")
    
    print("\n📊 Checking Feature Importance...")
    try:
        plot_feature_importance(last_model, X.columns)
    except Exception as e:
        print(f"Error plotting: {e}")

    return test_pred

---
# Part 3: Main Execution & Primary Submission

This is the **PRIMARY SUBMISSION** using single CatBoost model.

In [ ]:
# ==============================================================================
# MAIN EXECUTION - PRIMARY SUBMISSION
# ==============================================================================

print("\n" + "="*70)
print(" PHISHING URL DETECTION - LEAK-FREE PIPELINE")
print("="*70)

# 1. Load Data
X_cat, y, X_test, cat_cols = load_and_process_data()

# 2. Train & Predict
test_predictions = train_catboost_optimized(X_cat, y, X_test, cat_features=cat_cols, n_trials=10)

# 3. Save Primary Submission
sample_sub = pd.read_csv('/kaggle/input/dsco/sample_submission.csv')
sample_sub['ClassLabel'] = test_predictions
sample_sub.to_csv('submission_primary.csv', index=False)

print("\n" + "="*70)
print("✅ DONE! Primary submission saved as 'submission_primary.csv'")
print("="*70)

## Preview Primary Submission

In [ ]:
# Display first few rows of submission
submission = pd.read_csv('submission_primary.csv')
print("\n📄 Primary Submission File Preview:")
print(submission.head(10))
print(f"\nSubmission shape: {submission.shape}")
print(f"\nPrediction statistics:")
print(submission['ClassLabel'].describe())

---
# Part 4: Alternative Ensemble Approach (Optional)

This section provides an **alternative approach** using ensemble of multiple models:
- CatBoost
- XGBoost  
- LightGBM

Combined using:
1. **Simple weighted averaging** (Quick approach)
2. **Rank averaging** (Robust approach)
3. **Hill Climbing + SLSQP** (Best approach)


## Import Additional Libraries for Ensemble

In [ ]:
import xgboost as xgb
import lightgbm as lgb
from scipy.stats import rankdata
from sklearn.metrics import log_loss
from scipy.optimize import minimize

## Train Multiple Models for Ensemble

In [ ]:
def train_xgboost_model(X, y, X_test, n_trials=20):
    """
    Train XGBoost with Optuna and Fix Feature Mismatch (ID column)
    """
    # 1. ALIGN COLUMNS: Ensure X_test has exactly the same columns as X
    # This removes 'ID' and any extra columns from X_test automatically
    X = X.copy()
    X_test = X_test[X.columns].copy()
    
    # 2. CATEGORICAL CONVERSION
    cat_cols = X.select_dtypes(include=['object']).columns
    for col in cat_cols:
        X[col] = X[col].astype('category')
        X_test[col] = X_test[col].astype('category')
            
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    test_pred = np.zeros(len(X_test))
    oof_pred = np.zeros(len(X))
    
    print(f"\n🔎 Tuning XGBoost with Optuna ({n_trials} trials)...")
    
    # --- Optuna Objective ---
    def objective(trial):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 500, 2000),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'max_depth': trial.suggest_int('max_depth', 3, 10),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 7),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'reg_alpha': trial.suggest_float('reg_alpha', 0.01, 10.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 0.01, 10.0, log=True),
            'objective': 'binary:logistic',
            'eval_metric': 'auc',
            'random_state': 42,
            'verbosity': 0,
            'tree_method': 'hist',
            'enable_categorical': True
        }
        
        cv_scores = []
        temp_skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
        
        for train_idx, val_idx in temp_skf.split(X, y):
            X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
            
            model = xgb.XGBClassifier(**params)
            model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], early_stopping_rounds=50, verbose=False)
            cv_scores.append(roc_auc_score(y_val, model.predict_proba(X_val)[:, 1]))
            
        return np.mean(cv_scores)

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=n_trials)
    
    print(f"   Best XGB Params: {study.best_params}")
    best_params = study.best_params
    best_params.update({'objective': 'binary:logistic', 'eval_metric': 'auc', 
                        'random_state': 42, 'verbosity': 0, 'tree_method': 'hist', 'enable_categorical': True})

    # --- Final Training ---
    print("\n🚀 Training Final XGBoost with Best Params...")
    fold_scores = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        model = xgb.XGBClassifier(**best_params)
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], early_stopping_rounds=150, verbose=False)
        
        val_pred = model.predict_proba(X_val)[:, 1]
        oof_pred[val_idx] = val_pred
        score = roc_auc_score(y_val, val_pred)
        fold_scores.append(score)
        print(f"   Fold {fold+1}: AUC = {score:.6f}")
        
        test_pred += model.predict_proba(X_test)[:, 1] / skf.n_splits
    
    print(f"\n✅ XGBoost OOF AUC: {roc_auc_score(y, oof_pred):.6f}")
    return oof_pred, test_pred
    

def train_lightgbm_model(X, y, X_test, n_trials=20):
    """
    Train LightGBM with Optuna and Fix Feature Mismatch
    """
    # 1. ALIGN COLUMNS (Drop ID from Test)
    X = X.copy()
    X_test = X_test[X.columns].copy()
    
    # 2. CATEGORICAL CONVERSION
    cat_cols = X.select_dtypes(include=['object']).columns
    for col in cat_cols:
        X[col] = X[col].astype('category')
        X_test[col] = X_test[col].astype('category')

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    test_pred = np.zeros(len(X_test))
    oof_pred = np.zeros(len(X))
    
    print(f"\n🔎 Tuning LightGBM with Optuna ({n_trials} trials)...")
    
    # --- Optuna Objective ---
    def objective(trial):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 500, 2000),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'num_leaves': trial.suggest_int('num_leaves', 20, 150),
            'max_depth': trial.suggest_int('max_depth', 3, 12),
            'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'reg_alpha': trial.suggest_float('reg_alpha', 0.01, 10.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 0.01, 10.0, log=True),
            'objective': 'binary', 'metric': 'auc', 'random_state': 42, 'verbose': -1
        }
        
        cv_scores = []
        temp_skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
        for train_idx, val_idx in temp_skf.split(X, y):
            X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
            
            model = lgb.LGBMClassifier(**params)
            model.fit(X_tr, y_tr, eval_set=(X_val, y_val), callbacks=[lgb.early_stopping(50, verbose=False)])
            cv_scores.append(roc_auc_score(y_val, model.predict_proba(X_val)[:, 1]))
            
        return np.mean(cv_scores)

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=n_trials)
    
    print(f"   Best LGBM Params: {study.best_params}")
    best_params = study.best_params
    best_params.update({'objective': 'binary', 'metric': 'auc', 'random_state': 42, 'verbose': -1})

    # --- Final Training ---
    print("\n🚀 Training Final LightGBM with Best Params...")
    fold_scores = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        model = lgb.LGBMClassifier(**best_params)
        model.fit(X_tr, y_tr, eval_set=(X_val, y_val), callbacks=[lgb.early_stopping(150, verbose=False)])
        
        val_pred = model.predict_proba(X_val)[:, 1]
        oof_pred[val_idx] = val_pred
        score = roc_auc_score(y_val, val_pred)
        fold_scores.append(score)
        print(f"   Fold {fold+1}: AUC = {score:.6f}")
        
        test_pred += model.predict_proba(X_test)[:, 1] / skf.n_splits
    
    print(f"\n✅ LightGBM OOF AUC: {roc_auc_score(y, oof_pred):.6f}")
    return oof_pred, test_pred


def train_catboost_for_ensemble(X, y, X_test, cat_features=None, n_trials=15):
    """
    Train CatBoost with Optuna Hyperparameter Tuning for the Ensemble.
    """
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    test_pred = np.zeros(len(X_test))
    oof_pred = np.zeros(len(X))
    
    print(f"\n🔎 Tuning CatBoost with Optuna ({n_trials} trials)...")
    
    # --- 1. Optuna Objective ---
    def objective(trial):
        params = {
            'iterations': trial.suggest_int('iterations', 1000, 3000),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'depth': trial.suggest_int('depth', 4, 10),
            'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-3, 10.0, log=True),
            'random_strength': trial.suggest_float('random_strength', 1e-9, 10.0, log=True),
            'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
            # Fixed params
            'loss_function': 'Logloss',
            'eval_metric': 'AUC',
            'verbose': False,
            'random_seed': 42,
            'allow_writing_files': False,
            'task_type': 'CPU' # Change to 'GPU' if available
        }
        
        # Use a smaller subset or fewer folds for faster tuning
        temp_skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
        cv_scores = []
        
        for train_idx, val_idx in temp_skf.split(X, y):
            X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
            
            model = CatBoostClassifier(**params)
            model.fit(X_tr, y_tr, 
                      eval_set=(X_val, y_val), 
                      cat_features=cat_features, 
                      early_stopping_rounds=50, 
                      verbose=False)
            
            # Predict
            pred = model.predict_proba(X_val)[:, 1]
            score = roc_auc_score(y_val, pred)
            cv_scores.append(score)

        return np.mean(cv_scores)

    # Run Optuna
    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=n_trials)
    
    print(f"   Best CatBoost Params: {study.best_params}")
    best_params = study.best_params
    
    # Add static params back
    best_params.update({
        'loss_function': 'Logloss',
        'eval_metric': 'AUC',
        'verbose': False,
        'random_seed': 42,
        'allow_writing_files': False,
        'task_type': 'CPU'
    })

    # --- 2. Final Training with Best Params ---
    print("\n🚀 Training Final CatBoost with Best Params...")
    
    fold_scores = []
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        model = CatBoostClassifier(**best_params)
        model.fit(X_tr, y_tr, 
                  eval_set=(X_val, y_val), 
                  cat_features=cat_features, 
                  early_stopping_rounds=150, 
                  verbose=False)
        
        val_pred = model.predict_proba(X_val)[:, 1]
        oof_pred[val_idx] = val_pred
        score = roc_auc_score(y_val, val_pred)
        fold_scores.append(score)
        print(f"   Fold {fold+1}: AUC = {score:.6f}")
        
        test_pool = Pool(X_test, cat_features=cat_features)
        test_pred += model.predict_proba(test_pool)[:, 1] / skf.n_splits
    
    oof_auc = roc_auc_score(y, oof_pred)
    print(f"\n✅ CatBoost Mean CV Score: {np.mean(fold_scores):.6f}")
    print(f"✅ CatBoost OOF AUC: {oof_auc:.6f}")
    
    return oof_pred, test_pred

## Train All Models

In [ ]:
print("\n" + "="*70)
print(" TRAINING ENSEMBLE MODELS (WITH OPTUNA)")
print("="*70)

# 1. Train CatBoost (Native categorical support)
oof_cb, test_cb = train_catboost_for_ensemble(X_cat, y, X_test, cat_features=cat_cols, n_trials=50)

# 2. Train XGBoost (Converts object -> category internally in function)
oof_xgb, test_xgb = train_xgboost_model(X_cat, y, X_test, n_trials=50)

# 3. Train LightGBM (Converts object -> category internally in function)
oof_lgb, test_lgb = train_lightgbm_model(X_cat, y, X_test, n_trials=50)

## Ensemble Strategy 1: Simple Weighted Average

In [ ]:
print("\n" + "="*70)
print(" ENSEMBLE STRATEGY 1: WEIGHTED AVERAGE")
print("="*70)

# Calculate individual model AUC scores
auc_cb = roc_auc_score(y, oof_cb)
auc_xgb = roc_auc_score(y, oof_xgb)
auc_lgb = roc_auc_score(y, oof_lgb)

print(f"\nIndividual Model Scores:")
print(f"CatBoost:  {auc_cb:.6f}")
print(f"XGBoost:   {auc_xgb:.6f}")
print(f"LightGBM:  {auc_lgb:.6f}")

# Simple weighted average based on performance
total_score = auc_cb + auc_xgb + auc_lgb
w_cb = auc_cb / total_score
w_xgb = auc_xgb / total_score
w_lgb = auc_lgb / total_score

print(f"\nWeights based on performance:")
print(f"CatBoost:  {w_cb:.4f}")
print(f"XGBoost:   {w_xgb:.4f}")
print(f"LightGBM:  {w_lgb:.4f}")

# Create weighted ensemble
oof_weighted = w_cb * oof_cb + w_xgb * oof_xgb + w_lgb * oof_lgb
test_weighted = w_cb * test_cb + w_xgb * test_xgb + w_lgb * test_lgb

ensemble_auc_weighted = roc_auc_score(y, oof_weighted)
print(f"\n✅ ENSEMBLE (Weighted Average) OOF AUC: {ensemble_auc_weighted:.6f}")

## Ensemble Strategy 2: Rank Averaging

In [ ]:
print("\n" + "="*70)
print(" ENSEMBLE STRATEGY 2: RANK AVERAGING")
print("="*70)

# Convert predictions to ranks
oof_rank_cb = rankdata(oof_cb) / len(oof_cb)
oof_rank_xgb = rankdata(oof_xgb) / len(oof_xgb)
oof_rank_lgb = rankdata(oof_lgb) / len(oof_lgb)

test_rank_cb = rankdata(test_cb) / len(test_cb)
test_rank_xgb = rankdata(test_xgb) / len(test_xgb)
test_rank_lgb = rankdata(test_lgb) / len(test_lgb)

# Average ranks
oof_rank = (oof_rank_cb + oof_rank_xgb + oof_rank_lgb) / 3
test_rank = (test_rank_cb + test_rank_xgb + test_rank_lgb) / 3

ensemble_auc_rank = roc_auc_score(y, oof_rank)
print(f"\n✅ ENSEMBLE (Rank Average) OOF AUC: {ensemble_auc_rank:.6f}")

## Ensemble Strategy 3: Hill Climbing + SLSQP Ensemble

In [ ]:
print("\n" + "="*70)
print(" ENSEMBLE STRATEGY 3: HILL CLIMBING + SLSQP")
print("="*70)

# Prepare OOF and Test predictions from all models
models_oof = [oof_cb, oof_xgb, oof_lgb]
models_test = [test_cb, test_xgb, test_lgb]
model_names = ["CatBoost", "XGBoost", "LightGBM"]

# Calculate log loss for each model
scores = [log_loss(y, pred) for pred in models_oof]
sorted_idx = np.argsort(scores)

# Start with best single model
best_oof_pred = models_oof[sorted_idx[0]].copy()
best_test_pred = models_test[sorted_idx[0]].copy()
best_loss = scores[sorted_idx[0]]

print(f"\nStarting hill climbing with best single model: {model_names[sorted_idx[0]]} (logloss={best_loss:.6f})")

# Hill climbing: iteratively add models
for i in sorted_idx[1:]:
    stack_oof = np.column_stack([best_oof_pred, models_oof[i]])
    stack_test = np.column_stack([best_test_pred, models_test[i]])
    
    # SLSQP optimization to find best weights
    def objective(w):
        pred = np.clip(np.dot(stack_oof, w), 1e-15, 1-1e-15)
        return log_loss(y, pred)
    
    res = minimize(
        objective, 
        x0=[0.7, 0.3], 
        method='SLSQP',
        bounds=[(0.0, 1.0), (0.0, 1.0)],
        constraints={'type': 'eq', 'fun': lambda w: np.sum(w) - 1}
    )
    
    new_oof = np.clip(np.dot(stack_oof, res.x), 1e-15, 1-1e-15)
    new_loss = log_loss(y, new_oof)
    
    if new_loss < best_loss:
        print(f"✅ Improved by adding {model_names[i]} → new logloss = {new_loss:.6f} (weights: {res.x.round(4)})")
        best_loss = new_loss
        best_oof_pred = new_oof
        best_test_pred = np.dot(stack_test, res.x)
    else:
        print(f"❌ No improvement with {model_names[i]}")

# Calculate final metrics
final_auc = roc_auc_score(y, best_oof_pred)
print(f"\n✅ FINAL HILL CLIMBING ENSEMBLE OOF AUC     = {final_auc:.6f}")
print(f"✅ FINAL HILL CLIMBING ENSEMBLE OOF LOGLOSS = {best_loss:.6f}")

## Save Ensemble Submissions

In [ ]:
print("\n" + "="*70)
print(" SAVING ENSEMBLE SUBMISSIONS")
print("="*70)

sample_sub = pd.read_csv('/kaggle/input/dsco/sample_submission.csv')

# Save weighted average ensemble
sample_sub['ClassLabel'] = test_weighted
sample_sub.to_csv('submission_ensemble_weighted.csv', index=False)
print("✅ Saved: submission_ensemble_weighted.csv")

# Save rank average ensemble
sample_sub['ClassLabel'] = test_rank
sample_sub.to_csv('submission_ensemble_rank.csv', index=False)
print("✅ Saved: submission_ensemble_rank.csv")

# Save Hill Climbing submission
sample_sub['ClassLabel'] = best_test_pred
sample_sub.to_csv('submission_ensemble_hillclimb_slsqp.csv', index=False)
print("\n✅ Saved: submission_ensemble_hillclimb_slsqp.csv")

print("\n" + "="*70)
print(" ENSEMBLE COMPLETE!")
print("="*70)

print("\n📊 Summary of Results:")
print(f"Single CatBoost:                        {auc_cb:.6f}")
print(f"Single XGBoost:                         {auc_xgb:.6f}")
print(f"Single LightGBM:                        {auc_lgb:.6f}")
print(f"Ensemble (Weighted):                    {ensemble_auc_weighted:.6f}")
print(f"Ensemble (Rank):                        {ensemble_auc_rank:.6f}")
print(f"Ensemble (Hill Climbing + SLSQP):       {final_auc:.6f}")

---
## Summary

This notebook implements a complete pipeline for phishing URL detection with two approaches:

### Primary Approach (Part 3):
1. **Feature Extraction**: Network and content features from URLs
2. **Feature Engineering**: 40+ engineered features with leak-free fit/transform
3. **Single Model**: CatBoost with Optuna tuning and 5-fold CV
4. **Output**: `submission_primary.csv`

### Alternative Ensemble Approach (Part 4):
1. **Multiple Models**: CatBoost, XGBoost, LightGBM
2. **Ensemble Methods**:
   - Weighted averaging based on performance
   - Rank averaging for robustness
3. **Outputs**: 
   - `submission_ensemble_weighted.csv`
   - `submission_ensemble_rank.csv`
   - `submission_ensemble_hillclimb_slsqp.csv`

**Key Features:**
- No data leakage between train and test
- Resume capability for feature extraction
- Multiple ensemble strategies
- ROC-AUC optimization